In [10]:
import logging
import os

import cv2
import numpy as np
import psycopg2
from skimage.feature import graycomatrix, graycoprops

# Configuration log
logging.basicConfig(level=logging.INFO)

# Setting the root directory for image storage
image_root_dir = os.getenv("IMAGE_ROOT_DIR", "/Users/Tommy/AI/image-search/clothing-images/")

# image_root_dir = os.getenv("IMAGE_ROOT_DIR",
#                            "/Users/Tommy/MSC-TUD/Team_Project/dataset/image/zalando-hd-resized/test/cloth")

db_config = {
    'dbname': os.getenv('DB_NAME', 'image-search'),
    'user': os.getenv('DB_USER', 'postgres'),
    'password': os.getenv('DB_PASSWORD', 'test-postgres'),
    'host': os.getenv('DB_HOST', '127.0.0.1'),
}

batch_size = int(os.getenv("BATCH_SIZE", 100))

# database connection
conn = psycopg2.connect(**db_config)
cur = conn.cursor()

# Batch data storage
batch_data = []

# Iterate through the image catalog
for subdir, _, files in os.walk(image_root_dir):
    for filename in files:
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(subdir, filename)
            image_name = filename

            try:
                # Read image
                image = cv2.imread(image_path)
                if image is None:
                    logging.warning(f"Failed to load image: {image_path}")
                    continue

                # 1. color histogram
                hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
                color_hist = cv2.calcHist([hsv_image], [0, 1, 2], None, [4, 4, 4], [0, 256, 0, 256, 0, 256])
                original_color_hist = cv2.normalize(color_hist, color_hist).flatten()

                color_hist_2_db = original_color_hist.tolist()
                # print("color_hist: ", color_hist)

                # 2. GLCM Texture characteristics
                gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                distances = [1, 2, 3]  # Use multiple distances
                angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]  # Use multiple directions.
                glcm = graycomatrix(gray_image.astype(np.uint8), distances, angles)
                contrast = graycoprops(glcm, 'contrast')
                correlation = graycoprops(glcm, 'correlation')
                original_texture_features = np.hstack((contrast.flatten(), correlation.flatten()))

                texture_features = original_texture_features.tolist()
                # print("texture_features: ", texture_features)

                # 3. Perform Canny edge detection
                edges = cv2.Canny(gray_image, 50, 150)
                original_edge_density = np.mean(edges)  # Edge Pixel Density

                edge_density = [original_edge_density]
                # print("edge_density: ", edge_density)

                # Add to Batch
                batch_data.append((image_name, color_hist_2_db, texture_features, edge_density))

                # If batch size is reached, insert into database
                if len(batch_data) >= batch_size:
                    cur.executemany("""
                        INSERT INTO image_lowlevel_features_temp_1
                        (image_name, color_histogram, texture_features, edge_features)
                        VALUES (%s, %s, %s, %s)
                    """, batch_data)
                    conn.commit()
                    logging.info(f"Inserted {len(batch_data)} records into database.")
                    batch_data = []

            except Exception as e:
                logging.error(f"Error processing image {image_name}: {e}")

# Insert remaining data
if batch_data:
    cur.executemany("""
        INSERT INTO image_lowlevel_features_temp_1
        (image_name, color_histogram, texture_features, edge_features)
        VALUES (%s, %s, %s, %s)
    """, batch_data)
    conn.commit()
    logging.info(f"Inserted remaining {len(batch_data)} records into database.")

# Close the database connection
cur.close()
conn.close()
logging.info("All images processed and database connection closed.")


INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records int